## IMPORTACION DE LIBRERIAS
- Numpy libreria que facilita el trabajar con arrays permitiendonos operaciones numericas, algebra linea y multiplicacion de vectores

In [1]:
import numpy as np

## CARGA DEL DATASET CON NUMPY

Para la lectura del archivo `ventas_techventas.csv` se utilizó la función `np.genfromtxt()`, la cual permite cargar datos desde archivos de texto o csv y convertirlos directamente en arreglos de NumPy.

Se seleccionaron únicamente las columnas correspondientes a **cantidad** y **precio_unitario**, utilizando el parámetro `usecols`, ya que son los campos necesarios para los cálculos posteriores.

El parámetro `unpack=True` se empleó para separar automáticamente cada columna en un vector independiente, facilitando su manipulación y análisis.

Además, se configuró la carga para manejar posibles errores en los datos:

- `invalid_raise=False`: evita que el programa se detenga si encuentra filas con formatos incorrectos.
- `filling_values=np.nan`: reemplaza los valores faltantes por `NaN` (*Not a Number*), permitiendo identificarlos y tratarlos posteriormente durante el proceso de limpieza de datos.

El uso de `genfromtxt()` proporciona una forma segura y flexible de importar información, ya que permite trabajar incluso cuando el archivo contiene celdas vacías o registros incompletos.

In [2]:

cantidad, precio_unitario = np.genfromtxt(
    'ventas_techventas.csv',
    delimiter=',',
    skip_header=1,
    usecols=(7, 8),
    unpack=True,
    invalid_raise=False, # Si hay una fila con columnas rotas, la ignora en vez de romper el código
    filling_values=np.nan # Si una celda está vacía, la llena con NaN para filtrarla después
)


## Limpieza y validación de datos

Una vez cargados los datos, se realizó un proceso de limpieza para garantizar la calidad de la información antes de efectuar los cálculos estadísticos y las operaciones de álgebra lineal.

### Eliminación de valores nulos (NaN)

Se utilizó la función `np.isnan()` para identificar registros con valores nulos (`NaN`) en las columnas de cantidad y precio unitario. Posteriormente, se aplicó el operador `~` para conservar únicamente los registros que contienen información válida.

La eliminación de estos valores es importante porque los datos faltantes pueden generar resultados incorrectos o impedir la realización de ciertos cálculos matemáticos.

### Validación de valores negativos

También se filtraron los registros que contenían cantidades o precios menores o iguales a cero.

Aunque en algunos contextos los valores negativos pueden ser válidos, en este caso el conjunto de datos representa ventas de productos, por lo que una cantidad negativa o un precio negativo no tienen sentido desde el punto de vista del negocio.

Por ejemplo:

- No es posible vender `-1` producto.
- No es coherente registrar un precio de `-300.000`.
- Estos valores suelen ser consecuencia de errores de captura, digitación o problemas en la generación de los datos.

### Aplicación de la máscara de validación

La máscara de validación se aplicó simultáneamente a los vectores de cantidad y precio unitario para mantener la correspondencia entre ambos campos.

De esta manera, cada cantidad conserva su precio asociado y se evita la pérdida de consistencia en los registros.

### Tratamiento de registros duplicados

Los registros duplicados no fueron eliminados, ya que no necesariamente representan un error.

En un contexto de ventas es completamente posible que varios clientes compren la misma cantidad de productos al mismo precio. Por esta razón, los valores repetidos fueron considerados válidos y se mantuvieron dentro del análisis.

In [3]:

mascara_validos = (
    ~np.isnan(cantidad) &
    ~np.isnan(precio_unitario) &
    (cantidad > 0) &
    (precio_unitario > 0)
)

cantidad_limpia = cantidad[mascara_validos]
precio_limpio = precio_unitario[mascara_validos]

## CALCULO DE INGRESOS MEDIANTE ALGEBRA LINEAL

Para calcular el ingreso total generado por las ventas, se utilizó el producto punto entre los vectores de cantidades y precios unitarios mediante la función `np.dot()` de NumPy.

Cada posición del vector de cantidades corresponde a la misma posición del vector de precios unitarios. Por lo tanto, al multiplicar ambos vectores y sumar los resultados obtenidos, se calcula el ingreso total de todas las ventas registradas.

Matemáticamente, la operación puede representarse como:

Ingreso Total = (cantidad₁ × precio₁) + (cantidad₂ × precio₂) + ... + (cantidadₙ × precioₙ)

La función `np.dot()` realiza esta operación de forma eficiente, permitiendo aplicar conceptos de álgebra lineal al análisis de datos.

### Multiplicación elemento a elemento

Además del producto punto, se realizó una multiplicación elemento a elemento entre los vectores de cantidad y precio unitario.

Esta operación permite obtener el ingreso individual de cada registro de venta:

```python
ingresos_por_fila = cantidad_limpia * precio_limpio
```

Por ejemplo:

| Cantidad | Precio Unitario | Ingreso |
|-----------|----------------|----------|
| 2 | 1200 | 2400 |
| 3 | 500 | 1500 |
| 5 | 100 | 500 |

La multiplicación elemento a elemento resulta útil para analizar cada venta de manera individual, mientras que el producto punto (`np.dot`) permite obtener directamente el ingreso total del conjunto de datos.

In [4]:

ingreso_total = np.dot(cantidad_limpia, precio_limpio)
print(f"Ingreso bruto total limpio: {ingreso_total:.2f}")

ingresos_por_fila = cantidad_limpia * precio_limpio

Ingreso bruto total limpio: 89405.00


In [5]:
media = np.mean(precio_limpio)
mediana = np.median(precio_limpio)
desviacion = np.std(precio_limpio)
p25, p75 = np.percentile(precio_limpio, [25, 75])

print("Media:", media)
print("Mediana:", mediana)
print("Desviación estándar:", desviacion)
print("Percentil 25:", p25)
print("Percentil 75:", p75)

Media: 364.3220338983051
Mediana: 120.0
Desviación estándar: 435.6560002243243
Percentil 25: 90.0
Percentil 75: 350.0


## JUSTIFICACION DE LA LIMPIEZA DE DATOS

Antes de realizar el análisis estadístico, se llevó a cabo un proceso de limpieza de datos para eliminar valores nulos (`NaN`) y registros con valores negativos en las columnas de cantidad y precio unitario.

Esta depuración permite garantizar la calidad de la información utilizada en los cálculos, evitando que datos erróneos o incompletos afecten los resultados obtenidos.

Las métricas estadísticas (media, mediana, desviación estándar y percentiles 25 y 75) fueron calculadas únicamente sobre los registros válidos, proporcionando una representación más precisa del comportamiento real del catálogo de productos.

# Porque usamos estas metricas juntas?

| Métrica             | Qué muestra             |
| ------------------- | ----------------------- |
| Media               | Precio promedio         |
| Mediana             | Precio central          |
| Desviación estándar | Variabilidad de precios |
| Percentil 25        | Zona de precios bajos   |
| Percentil 75        | Zona de precios altos   |


## INTERPRETACION DE LOS RESULTADOS ESTADISTICOS

### Media

La media obtenida fue de **364.32**.

Esto significa que, en promedio, los productos del catálogo tienen un precio aproximado de 364 unidades monetarias. Sin embargo, la media puede verse afectada por productos con precios muy elevados.

### Mediana

La mediana fue de **120.00**.

Este resultado indica que el 50% de los productos tiene un precio menor o igual a 120, mientras que el otro 50% tiene un precio mayor o igual a este valor.

La diferencia entre la media (364.32) y la mediana (120.00) sugiere la presencia de productos con precios significativamente más altos que elevan el promedio general.

### Desviación estándar

La desviación estándar fue de **435.66**.

Este valor relativamente alto indica que existe una gran dispersión en los precios de los productos, es decir, los precios no se encuentran concentrados alrededor de un único valor, sino que presentan una amplia variabilidad.

### Percentil 25

El percentil 25 fue de **90.00**.

Esto significa que el 25% de los productos tiene un precio menor o igual a 90.

### Percentil 75

El percentil 75 fue de **350.00**.

Esto significa que el 75% de los productos tiene un precio menor o igual a 350.

### Conclusión

Los resultados muestran que la distribución de precios no es uniforme. La diferencia entre la media y la mediana, junto con la alta desviación estándar, indica la existencia de productos con precios considerablemente superiores al resto del catálogo.

Además, el hecho de que el 50% de los productos cueste 120 o menos, mientras que el promedio general sea 364.32, refuerza la hipótesis de que existen valores altos que influyen significativamente en la media.